# SI Figure S14: test RMSE with DFT vs MagNET features, by solvent

For the four explicit-solvent solvents (chloroform, methanol, TIP4P water, benzene), per-solvent test
RMSE of predicting experimental ¹H/¹³C shifts with **DFT** features (solid) vs end-to-end **MagNET/NN**
features (lightened), each under three conditions (implicit SOTA, explicit, explicit + vibrations).
Two pages, one per MD engine (Desmond, OpenMM); nitromethane dropped.

In [ ]:
import os, sys

# make the in-repo modules importable (not pip-installed)
REPO = os.path.abspath("../..")
for _p in ("data/delta22", "analysis/code", "analysis/code/shared"):
    sys.path.insert(0, os.path.join(REPO, _p))

In [ ]:
import matplotlib.pyplot as plt

import delta22
import delta22_plots
import paths

In [ ]:
DELTA22_HDF5 = paths.dataset_file("delta22", root=REPO)
XLSX = os.path.join(REPO, "data", "delta22", "delta22_experimental.xlsx")

def figure_path(name):
    os.makedirs("figures", exist_ok=True)
    return os.path.join("figures", name)

In [ ]:
# the published panels use 250 seeded train/test splits
N_SPLITS = 250

In [ ]:
dft = delta22.add_composite_columns(delta22.load_query_df_dft(DELTA22_HDF5, XLSX, verbose=False))
nn = delta22.add_composite_columns(delta22.load_query_df_nn(DELTA22_HDF5, XLSX, verbose=False))
print(len(dft), "DFT rows;", len(nn), "NN rows")

LABELS = ["Implicit Solvent (SOTA)", "Explicit Solvent", "Explicit Solvent + Vibrations"]
SS_LABELS = {"TIP4P": "Water (TIP4P)"}
S14_SOLVENTS = ["chloroform", "methanol", "TIP4P", "benzene"]   # the 4 explicit solvents, SI order

## Desmond page (DFT vs NN)

In [ ]:
# implicit = 2-term (stationary + pcm); explicit = stationary + desmond;
# explicit + vibrations = stationary_plus_qcd + desmond (1H) / stationary_plus_des_vib + desmond (13C)
DESMOND_FORMULAS = {
    "H":  ["stationary + pcm", "stationary + desmond", "stationary_plus_qcd + desmond"],
    "C":  ["stationary + pcm", "stationary + desmond", "stationary_plus_des_vib + desmond"],
}
for nucleus, label in [("H", "1H"), ("C", "13C")]:
    formulas = DESMOND_FORMULAS[nucleus]
    delta22_plots.plot_ss_boxplot_dft_vs_nn(
        delta22_plots.ss_fits(dft, nucleus, formulas, S14_SOLVENTS, N_SPLITS, dft=True),
        delta22_plots.ss_fits(nn, nucleus, formulas, S14_SOLVENTS, N_SPLITS),
        S14_SOLVENTS, formulas, LABELS, label, solvent_labels=SS_LABELS,
        save_path=figure_path(f"si_figure_s14_desmond_{label}.png"))
plt.show()

## OpenMM page (DFT vs NN)

In [ ]:
# implicit = 1-term composite (stationary_plus_pcm); explicit = stationary + openMM;
# explicit + vibrations = stationary_plus_qcd + openMM (1H) / stationary_plus_op_vib + openMM (13C)
OPENMM_FORMULAS = {
    "H":  ["stationary_plus_pcm", "stationary + openMM", "stationary_plus_qcd + openMM"],
    "C":  ["stationary_plus_pcm", "stationary + openMM", "stationary_plus_op_vib + openMM"],
}
for nucleus, label in [("H", "1H"), ("C", "13C")]:
    formulas = OPENMM_FORMULAS[nucleus]
    delta22_plots.plot_ss_boxplot_dft_vs_nn(
        delta22_plots.ss_fits(dft, nucleus, formulas, S14_SOLVENTS, N_SPLITS, dft=True),
        delta22_plots.ss_fits(nn, nucleus, formulas, S14_SOLVENTS, N_SPLITS),
        S14_SOLVENTS, formulas, LABELS, label, solvent_labels=SS_LABELS,
        save_path=figure_path(f"si_figure_s14_openmm_{label}.png"))
plt.show()